# Gemma Runtime Smoke-Test Harness

This notebook tests Google Gemma models across three runtimes:
- **Transformers** (Hugging Face, in-notebook Python)
- **Ollama** (external CLI/API)
- **llamafile** (single-file distribution)

## Goals
1. Confirm model runnability across different hardware configurations
2. Collect reliable, comparable timings
3. Provide clear debug signals for failures
4. Generate actionable recommendations for local/sovereign agentic apps

## Models Tested
- Gemma 3: 1b, 4b, 12b (with Gemma 2 fallbacks)
- All models are instruction-tuned variants (-it)

## Output
- `artifacts/results.jsonl` - Raw test results
- `artifacts/results.csv` - Flattened summary
- `artifacts/report.md` - OpenAI-generated analysis
- `artifacts/report.json` - Machine-readable report

## Setup: Imports and Environment

In [ ]:
import os
import sys
import json
import time
import subprocess
import platform
import psutil
import warnings
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Any, Tuple
from getpass import getpass
import pandas as pd

warnings.filterwarnings('ignore')

# Create output directory
ARTIFACTS_DIR = Path('artifacts')
ARTIFACTS_DIR.mkdir(exist_ok=True)

print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"CPU Count: {psutil.cpu_count()}")
print(f"RAM: {psutil.virtual_memory().total / (1024**3):.1f} GB")

## Configuration Section

All user-configurable parameters are defined here.

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

# Runtime flags - which tests to run
RUN_TRANSFORMERS = True
RUN_OLLAMA = True
RUN_LLAMAFILE = True
RUN_OPENAI_REPORT = True

# Model lists
TRANSFORMERS_MODELS = [
    "google/gemma-3-1b-it",
    "google/gemma-3-4b-it",
    "google/gemma-3-12b-it",
]

TRANSFORMERS_FALLBACK_MODELS = [
    "google/gemma-2-2b-it",
    "google/gemma-2-9b-it",
]

OLLAMA_MODELS = [
    "gemma3:1b",
    "gemma3:4b",
    "gemma3:12b",
]

OLLAMA_FALLBACK_MODELS = [
    "gemma2:2b",
    "gemma2:9b",
]

LLAMAFILE_MODELS = [
    "mozilla-ai/gemma-3-1b-it-llamafile",
    "mozilla-ai/gemma-3-4b-it-llamafile",
    "mozilla-ai/gemma-3-12b-it-llamafile",
]

LLAMAFILE_FALLBACK_MODELS = [
    "mozilla-ai/gemma-2-2b-it-llamafile",
]

# Timeouts (in seconds)
DOWNLOAD_TIMEOUT_S = 1800  # 30 minutes for large models
FIRST_LOAD_TIMEOUT_S = 600  # 10 minutes for first load
INFERENCE_TIMEOUT_S = 120  # 2 minutes per prompt
PROCESS_SHUTDOWN_TIMEOUT_S = 30  # 30 seconds for cleanup

# Generation parameters
MAX_TOKENS = 512
TEMPERATURE = 0.7
SEED = 42

# Prompts
SHORT_PROMPT = "What is the capital of France?"

COMPLEX_PROMPT = """Extract information from the following text and output as JSON with keys 'name', 'age', 'city', and 'occupation':

Text: John Smith is a 35-year-old software engineer living in San Francisco.

Output the result as valid JSON only, no additional text."""

PROMPTS = [
    {"name": "short", "text": SHORT_PROMPT},
    {"name": "complex", "text": COMPLEX_PROMPT},
]

# OpenAI configuration
OPENAI_MODEL = "gpt-4o"  # Use gpt-4o or latest available

# Output paths
RESULTS_JSONL = ARTIFACTS_DIR / "results.jsonl"
RESULTS_CSV = ARTIFACTS_DIR / "results.csv"
REPORT_MD = ARTIFACTS_DIR / "report.md"
REPORT_JSON = ARTIFACTS_DIR / "report.json"

print("✓ Configuration loaded")

## Token Management

Prompt for required API tokens if not set in environment.

In [ ]:
# Hugging Face token
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN and RUN_TRANSFORMERS:
    print("Hugging Face token not found in environment.")
    HF_TOKEN = getpass("Enter HF_TOKEN (or press Enter to skip Transformers tests): ")
    if not HF_TOKEN:
        print("⚠ No HF token provided. Transformers tests will be skipped if gated models are encountered.")
        RUN_TRANSFORMERS = False
    else:
        os.environ['HF_TOKEN'] = HF_TOKEN
        print("✓ HF token set")

# OpenAI API key (always prompt, don't use env)
OPENAI_API_KEY = None
if RUN_OPENAI_REPORT:
    OPENAI_API_KEY = getpass("Enter OPENAI_API_KEY (or press Enter to skip report generation): ")
    if not OPENAI_API_KEY:
        print("⚠ No OpenAI API key provided. Report generation will be skipped.")
        RUN_OPENAI_REPORT = False
    else:
        print("✓ OpenAI API key set")

## Utility Functions

In [ ]:
def get_environment_fingerprint() -> Dict[str, Any]:
    """Collect environment metadata for reproducibility."""
    fingerprint = {
        "timestamp": datetime.now().isoformat(),
        "os": platform.platform(),
        "python_version": sys.version,
        "cpu_count": psutil.cpu_count(),
        "ram_gb": psutil.virtual_memory().total / (1024**3),
    }
    
    # Try to get package versions
    try:
        import torch
        fingerprint["torch_version"] = torch.__version__
        fingerprint["cuda_available"] = torch.cuda.is_available()
        if torch.cuda.is_available():
            fingerprint["cuda_version"] = torch.version.cuda
            fingerprint["gpu_name"] = torch.cuda.get_device_name(0)
        if hasattr(torch.backends, 'mps'):
            fingerprint["mps_available"] = torch.backends.mps.is_available()
    except ImportError:
        fingerprint["torch_version"] = "not_installed"
    
    try:
        import transformers
        fingerprint["transformers_version"] = transformers.__version__
    except ImportError:
        fingerprint["transformers_version"] = "not_installed"
    
    return fingerprint


def log_phase(phase: str, **kwargs):
    """Log a timestamped phase marker."""
    log_entry = {
        "timestamp": datetime.now().isoformat(),
        "phase": phase,
        **kwargs
    }
    print(json.dumps(log_entry))
    return log_entry


def get_memory_info() -> Dict[str, float]:
    """Get current memory usage."""
    vm = psutil.virtual_memory()
    return {
        "total_gb": vm.total / (1024**3),
        "available_gb": vm.available / (1024**3),
        "used_gb": vm.used / (1024**3),
        "percent": vm.percent,
    }


def clear_memory():
    """Aggressively clear memory."""
    log_phase("CLEAR_START", memory=get_memory_info())
    
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
    except ImportError:
        pass
    
    import gc
    gc.collect()
    
    log_phase("CLEAR_DONE", memory=get_memory_info())


def save_result(result: Dict[str, Any]):
    """Append result to JSONL file."""
    with open(RESULTS_JSONL, 'a') as f:
        f.write(json.dumps(result) + '\n')


def create_result_record(
    runtime: str,
    model: str,
    status: str,
    prompt_name: str = "",
    prompt_text: str = "",
    output: str = "",
    duration_s: float = 0.0,
    device: str = "",
    error: str = "",
    **kwargs
) -> Dict[str, Any]:
    """Create a standardized result record."""
    return {
        "timestamp": datetime.now().isoformat(),
        "runtime": runtime,
        "model": model,
        "status": status,  # SUCCESS, FAILED, SKIPPED
        "prompt_name": prompt_name,
        "prompt_text": prompt_text,
        "output": output,
        "duration_s": duration_s,
        "device": device,
        "error": error,
        **kwargs
    }

print("✓ Utility functions loaded")

## Environment Fingerprint

In [ ]:
env_fingerprint = get_environment_fingerprint()
print(json.dumps(env_fingerprint, indent=2))

## Transformers Runtime Tests

Test models using Hugging Face Transformers library.

**Success criteria:**
- Model loads successfully
- Inference completes within timeout
- Output is generated for each prompt
- Device info is logged (CPU, CUDA, MPS)

In [ ]:
def test_transformers_model(model_id: str) -> List[Dict[str, Any]]:
    """Test a single Transformers model with all prompts."""
    results = []
    model = None
    tokenizer = None
    
    try:
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM
        
        # Determine device
        if torch.cuda.is_available():
            device = "cuda"
        elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
            device = "mps"
        else:
            device = "cpu"
        
        log_phase("LOAD_START", model=model_id, device=device, memory=get_memory_info())
        
        # Load model
        start_time = time.time()
        try:
            tokenizer = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN)
            model = AutoModelForCausalLM.from_pretrained(
                model_id,
                token=HF_TOKEN,
                torch_dtype=torch.float16 if device == "cuda" else torch.float32,
                device_map="auto" if device == "cuda" else None,
            )
            if device != "cuda":
                model = model.to(device)
            model.eval()
            load_duration = time.time() - start_time
            log_phase("LOAD_DONE", model=model_id, duration_s=load_duration, memory=get_memory_info())
        except Exception as e:
            error_msg = f"{type(e).__name__}: {str(e)[:200]}"
            log_phase("LOAD_FAILED", model=model_id, error=error_msg)
            results.append(create_result_record(
                runtime="transformers",
                model=model_id,
                status="FAILED",
                error=error_msg,
                device=device,
            ))
            return results
        
        # Run inference for each prompt
        for prompt_info in PROMPTS:
            prompt_name = prompt_info["name"]
            prompt_text = prompt_info["text"]
            
            try:
                log_phase("INFER_START", model=model_id, prompt=prompt_name)
                
                start_time = time.time()
                inputs = tokenizer(prompt_text, return_tensors="pt").to(device)
                
                with torch.no_grad():
                    outputs = model.generate(
                        **inputs,
                        max_new_tokens=MAX_TOKENS,
                        temperature=TEMPERATURE,
                        do_sample=True,
                    )
                
                output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
                duration = time.time() - start_time
                
                log_phase("INFER_DONE", model=model_id, prompt=prompt_name, duration_s=duration)
                
                results.append(create_result_record(
                    runtime="transformers",
                    model=model_id,
                    status="SUCCESS",
                    prompt_name=prompt_name,
                    prompt_text=prompt_text,
                    output=output_text,
                    duration_s=duration,
                    device=device,
                ))
            except Exception as e:
                error_msg = f"{type(e).__name__}: {str(e)[:200]}"
                log_phase("INFER_FAILED", model=model_id, prompt=prompt_name, error=error_msg)
                results.append(create_result_record(
                    runtime="transformers",
                    model=model_id,
                    status="FAILED",
                    prompt_name=prompt_name,
                    prompt_text=prompt_text,
                    error=error_msg,
                    device=device,
                ))
    
    except ImportError as e:
        error_msg = f"Import error: {str(e)}"
        results.append(create_result_record(
            runtime="transformers",
            model=model_id,
            status="SKIPPED",
            error=error_msg,
        ))
    finally:
        # Cleanup
        del model
        del tokenizer
        clear_memory()
    
    return results


if RUN_TRANSFORMERS:
    print("\n" + "="*80)
    print("TRANSFORMERS RUNTIME TESTS")
    print("="*80 + "\n")
    
    # Try primary models first
    all_models = TRANSFORMERS_MODELS + TRANSFORMERS_FALLBACK_MODELS
    
    for model_id in all_models:
        print(f"\nTesting: {model_id}")
        results = test_transformers_model(model_id)
        for result in results:
            save_result(result)
            print(f"  {result['status']}: {result.get('prompt_name', 'N/A')} ({result.get('duration_s', 0):.2f}s)")
        print()
else:
    print("⚠ Transformers tests skipped")

## Ollama Runtime Tests

Test models using Ollama CLI.

**Success criteria:**
- Ollama is installed and server is reachable
- Model pulls successfully (or is already available)
- Inference completes within timeout
- Output is captured correctly

In [ ]:
def check_ollama_available() -> Tuple[bool, str]:
    """Check if Ollama is installed and server is running."""
    try:
        result = subprocess.run(
            ["ollama", "list"],
            capture_output=True,
            text=True,
            timeout=10,
        )
        if result.returncode == 0:
            return True, "Ollama is available"
        else:
            return False, f"Ollama CLI error: {result.stderr}"
    except FileNotFoundError:
        return False, "Ollama CLI not found"
    except subprocess.TimeoutExpired:
        return False, "Ollama CLI timeout"
    except Exception as e:
        return False, f"Ollama check error: {str(e)}"


def test_ollama_model(model_tag: str) -> List[Dict[str, Any]]:
    """Test a single Ollama model with all prompts."""
    results = []
    
    try:
        # Pull model if needed
        log_phase("DOWNLOAD_START", model=model_tag, runtime="ollama")
        pull_result = subprocess.run(
            ["ollama", "pull", model_tag],
            capture_output=True,
            text=True,
            timeout=DOWNLOAD_TIMEOUT_S,
        )
        
        if pull_result.returncode != 0:
            error_msg = f"Pull failed: {pull_result.stderr[:200]}"
            log_phase("DOWNLOAD_FAILED", model=model_tag, error=error_msg)
            results.append(create_result_record(
                runtime="ollama",
                model=model_tag,
                status="FAILED",
                error=error_msg,
            ))
            return results
        
        log_phase("DOWNLOAD_DONE", model=model_tag)
        
        # Run inference for each prompt
        for prompt_info in PROMPTS:
            prompt_name = prompt_info["name"]
            prompt_text = prompt_info["text"]
            
            try:
                log_phase("INFER_START", model=model_tag, prompt=prompt_name)
                
                start_time = time.time()
                result = subprocess.run(
                    ["ollama", "run", model_tag, prompt_text],
                    capture_output=True,
                    text=True,
                    timeout=INFERENCE_TIMEOUT_S,
                )
                duration = time.time() - start_time
                
                if result.returncode == 0:
                    output_text = result.stdout.strip()
                    log_phase("INFER_DONE", model=model_tag, prompt=prompt_name, duration_s=duration)
                    
                    results.append(create_result_record(
                        runtime="ollama",
                        model=model_tag,
                        status="SUCCESS",
                        prompt_name=prompt_name,
                        prompt_text=prompt_text,
                        output=output_text,
                        duration_s=duration,
                    ))
                else:
                    error_msg = f"Inference failed: {result.stderr[:200]}"
                    log_phase("INFER_FAILED", model=model_tag, prompt=prompt_name, error=error_msg)
                    results.append(create_result_record(
                        runtime="ollama",
                        model=model_tag,
                        status="FAILED",
                        prompt_name=prompt_name,
                        prompt_text=prompt_text,
                        error=error_msg,
                    ))
            except subprocess.TimeoutExpired:
                error_msg = f"Inference timeout after {INFERENCE_TIMEOUT_S}s"
                log_phase("INFER_FAILED", model=model_tag, prompt=prompt_name, error=error_msg)
                results.append(create_result_record(
                    runtime="ollama",
                    model=model_tag,
                    status="FAILED",
                    prompt_name=prompt_name,
                    prompt_text=prompt_text,
                    error=error_msg,
                ))
            except Exception as e:
                error_msg = f"{type(e).__name__}: {str(e)[:200]}"
                log_phase("INFER_FAILED", model=model_tag, prompt=prompt_name, error=error_msg)
                results.append(create_result_record(
                    runtime="ollama",
                    model=model_tag,
                    status="FAILED",
                    prompt_name=prompt_name,
                    prompt_text=prompt_text,
                    error=error_msg,
                ))
    except Exception as e:
        error_msg = f"{type(e).__name__}: {str(e)[:200]}"
        results.append(create_result_record(
            runtime="ollama",
            model=model_tag,
            status="FAILED",
            error=error_msg,
        ))
    finally:
        clear_memory()
    
    return results


if RUN_OLLAMA:
    print("\n" + "="*80)
    print("OLLAMA RUNTIME TESTS")
    print("="*80 + "\n")
    
    available, message = check_ollama_available()
    print(f"Ollama status: {message}\n")
    
    if available:
        all_models = OLLAMA_MODELS + OLLAMA_FALLBACK_MODELS
        
        for model_tag in all_models:
            print(f"\nTesting: {model_tag}")
            results = test_ollama_model(model_tag)
            for result in results:
                save_result(result)
                print(f"  {result['status']}: {result.get('prompt_name', 'N/A')} ({result.get('duration_s', 0):.2f}s)")
            print()
    else:
        print(f"⚠ Ollama tests skipped: {message}")
        # Save skipped record
        for model_tag in OLLAMA_MODELS:
            save_result(create_result_record(
                runtime="ollama",
                model=model_tag,
                status="SKIPPED",
                error=message,
            ))
else:
    print("⚠ Ollama tests skipped (disabled in config)")

## llamafile Runtime Tests

Test models using llamafile executables.

**Success criteria:**
- llamafile downloads successfully
- Executable permissions are set correctly
- Model runs and produces output
- Process terminates cleanly

In [ ]:
def download_llamafile(repo_id: str) -> Tuple[Optional[Path], str]:
    """Download a llamafile from Hugging Face."""
    try:
        from huggingface_hub import hf_hub_download
        
        # Extract model name from repo_id
        model_name = repo_id.split('/')[-1]
        
        # Look for .llamafile in the repo
        log_phase("DOWNLOAD_START", model=repo_id, runtime="llamafile")
        
        try:
            # Try to download the llamafile
            llamafile_path = hf_hub_download(
                repo_id=repo_id,
                filename=f"{model_name}.llamafile",
                token=HF_TOKEN,
                cache_dir=str(ARTIFACTS_DIR / "llamafile_cache"),
            )
            
            # Make executable
            import stat
            path = Path(llamafile_path)
            path.chmod(path.stat().st_mode | stat.S_IEXEC)
            
            log_phase("DOWNLOAD_DONE", model=repo_id)
            return path, ""
        except Exception as e:
            error_msg = f"Download failed: {str(e)[:200]}"
            log_phase("DOWNLOAD_FAILED", model=repo_id, error=error_msg)
            return None, error_msg
    except ImportError:
        return None, "huggingface_hub not installed"


def test_llamafile_model(repo_id: str) -> List[Dict[str, Any]]:
    """Test a single llamafile model with all prompts."""
    results = []
    
    # Download llamafile
    llamafile_path, error = download_llamafile(repo_id)
    if not llamafile_path:
        results.append(create_result_record(
            runtime="llamafile",
            model=repo_id,
            status="FAILED",
            error=error,
        ))
        return results
    
    # Test each prompt
    for prompt_info in PROMPTS:
        prompt_name = prompt_info["name"]
        prompt_text = prompt_info["text"]
        
        try:
            log_phase("INFER_START", model=repo_id, prompt=prompt_name)
            
            start_time = time.time()
            # Run llamafile with prompt
            result = subprocess.run(
                [str(llamafile_path), "-p", prompt_text, "--temp", str(TEMPERATURE), "-n", str(MAX_TOKENS)],
                capture_output=True,
                text=True,
                timeout=INFERENCE_TIMEOUT_S,
            )
            duration = time.time() - start_time
            
            if result.returncode == 0:
                output_text = result.stdout.strip()
                log_phase("INFER_DONE", model=repo_id, prompt=prompt_name, duration_s=duration)
                
                results.append(create_result_record(
                    runtime="llamafile",
                    model=repo_id,
                    status="SUCCESS",
                    prompt_name=prompt_name,
                    prompt_text=prompt_text,
                    output=output_text,
                    duration_s=duration,
                ))
            else:
                error_msg = f"Execution failed: {result.stderr[:200]}"
                log_phase("INFER_FAILED", model=repo_id, prompt=prompt_name, error=error_msg)
                results.append(create_result_record(
                    runtime="llamafile",
                    model=repo_id,
                    status="FAILED",
                    prompt_name=prompt_name,
                    prompt_text=prompt_text,
                    error=error_msg,
                ))
        except subprocess.TimeoutExpired:
            error_msg = f"Inference timeout after {INFERENCE_TIMEOUT_S}s"
            log_phase("INFER_FAILED", model=repo_id, prompt=prompt_name, error=error_msg)
            results.append(create_result_record(
                runtime="llamafile",
                model=repo_id,
                status="FAILED",
                prompt_name=prompt_name,
                prompt_text=prompt_text,
                error=error_msg,
            ))
        except Exception as e:
            error_msg = f"{type(e).__name__}: {str(e)[:200]}"
            log_phase("INFER_FAILED", model=repo_id, prompt=prompt_name, error=error_msg)
            results.append(create_result_record(
                runtime="llamafile",
                model=repo_id,
                status="FAILED",
                prompt_name=prompt_name,
                prompt_text=prompt_text,
                error=error_msg,
            ))
        finally:
            clear_memory()
    
    return results


if RUN_LLAMAFILE:
    print("\n" + "="*80)
    print("LLAMAFILE RUNTIME TESTS")
    print("="*80 + "\n")
    
    all_models = LLAMAFILE_MODELS + LLAMAFILE_FALLBACK_MODELS
    
    for repo_id in all_models:
        print(f"\nTesting: {repo_id}")
        results = test_llamafile_model(repo_id)
        for result in results:
            save_result(result)
            print(f"  {result['status']}: {result.get('prompt_name', 'N/A')} ({result.get('duration_s', 0):.2f}s)")
        print()
else:
    print("⚠ llamafile tests skipped")

## Results Summary

Load and display test results.

In [ ]:
# Load results
if RESULTS_JSONL.exists():
    results_data = []
    with open(RESULTS_JSONL, 'r') as f:
        for line in f:
            results_data.append(json.loads(line))
    
    # Create DataFrame
    df = pd.DataFrame(results_data)
    
    # Save CSV
    df.to_csv(RESULTS_CSV, index=False)
    print(f"✓ Saved results to {RESULTS_CSV}")
    
    # Display summary
    print("\n" + "="*80)
    print("RESULTS SUMMARY")
    print("="*80 + "\n")
    
    print("Status counts:")
    print(df['status'].value_counts())
    print()
    
    print("Results by runtime:")
    print(df.groupby(['runtime', 'status']).size())
    print()
    
    if len(df[df['status'] == 'SUCCESS']) > 0:
        print("Successful runs - average duration:")
        success_df = df[df['status'] == 'SUCCESS']
        print(success_df.groupby(['runtime', 'model'])['duration_s'].mean())
else:
    print("⚠ No results file found")

## OpenAI Report Generation

Generate an analysis report using OpenAI API.

**Success criteria:**
- Report analyzes which models were runnable
- Failures are clustered by type
- Practical recommendations are provided
- Best latency metrics are highlighted

In [ ]:
if RUN_OPENAI_REPORT and RESULTS_JSONL.exists():
    print("\n" + "="*80)
    print("GENERATING OPENAI REPORT")
    print("="*80 + "\n")
    
    try:
        from openai import OpenAI
        
        client = OpenAI(api_key=OPENAI_API_KEY)
        
        # Load all results
        results_data = []
        with open(RESULTS_JSONL, 'r') as f:
            for line in f:
                results_data.append(json.loads(line))
        
        # Prepare prompt for OpenAI
        prompt = f"""Analyze the following benchmark results from testing Google Gemma models across three runtimes (Transformers, Ollama, llamafile).

Environment:
{json.dumps(env_fingerprint, indent=2)}

Test Results:
{json.dumps(results_data, indent=2)}

Please provide a concise report (max 1000 words) that includes:

1. Executive Summary: Which models were runnable in this environment and which runtime(s) worked best?

2. Failure Analysis: Cluster failures by type (e.g., download/gating issues, OOM, runtime not installed, timeouts) and explain the root causes.

3. Performance Metrics: Create a table showing the best achieved latency (duration_s) and estimated tokens/sec for each successfully tested model and runtime.

4. Recommendations: For someone building "local, sovereign agentic apps" on this machine class:
   - Which runtime should they use?
   - Which model sizes are realistic?
   - What are the practical tradeoffs?

Format the response in Markdown."""
        
        print("Calling OpenAI API...")
        response = client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[
                {"role": "system", "content": "You are an expert in ML infrastructure and model deployment."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.3,
        )
        
        report_text = response.choices[0].message.content
        
        # Save markdown report
        with open(REPORT_MD, 'w') as f:
            f.write(report_text)
        print(f"✓ Saved report to {REPORT_MD}")
        
        # Save JSON report
        report_json = {
            "generated_at": datetime.now().isoformat(),
            "model_used": OPENAI_MODEL,
            "environment": env_fingerprint,
            "report_text": report_text,
            "results_summary": {
                "total_tests": len(results_data),
                "successful": len([r for r in results_data if r['status'] == 'SUCCESS']),
                "failed": len([r for r in results_data if r['status'] == 'FAILED']),
                "skipped": len([r for r in results_data if r['status'] == 'SKIPPED']),
            }
        }
        
        with open(REPORT_JSON, 'w') as f:
            json.dump(report_json, f, indent=2)
        print(f"✓ Saved report metadata to {REPORT_JSON}")
        
        # Display report
        print("\n" + "="*80)
        print("REPORT")
        print("="*80 + "\n")
        print(report_text)
        
    except Exception as e:
        print(f"⚠ Report generation failed: {type(e).__name__}: {str(e)}")
elif not RUN_OPENAI_REPORT:
    print("⚠ OpenAI report generation skipped (disabled in config)")
else:
    print("⚠ No results file found - skipping report generation")

## Test Complete

All tests have been executed. Check the `artifacts/` directory for:
- `results.jsonl` - Raw test results
- `results.csv` - Flattened summary
- `report.md` - Human-readable analysis
- `report.json` - Machine-readable report metadata